---
title: "DRG Cleaning"

author: "Carlos Resurreccion"

date: "2024-07-01"

---

In [1]:
knitr::opts_chunk$set(echo = TRUE)


## Load Required Libraries

In [2]:
options(verbose = FALSE)
options(warn = -1)


In [3]:
library(here)
source(here("data-cleaning", "r_scripts", "libraries.R"))
# source(here("data-cleaning", "r_scripts", "everything.R"))


here() starts at C:/Users/resur/Documents/drg-pipeline



In [4]:
options(warn = 1)


## Set Parameters

### Year to Load, Version, and Parameters

In [5]:
source(here("data-cleaning", "r_scripts", "parameters.R"))


## Source Data Formats

In [6]:
source(here("data-cleaning", "r_scripts", "data-formats.R"))
source(here("data-cleaning", "r_scripts", "file-paths.R"))


## Source Functions

In [7]:
source(here("data-cleaning", "r_scripts", "general-functions.R"))
# source(here("data-cleaning", "r_scripts", "clean-data-mini-functions.R"))
# source(here("data-cleaning", "r_scripts", "profvis.R"))
source(here("data-cleaning", "r_scripts", "main-functions.R"))
source(here("data-cleaning", "r_scripts", "icd-functions.R"))
source(here("data-cleaning", "r_scripts", "rvs-functions.R"))
source(here("data-cleaning", "r_scripts", "pdx-functions.R"))
source(here("data-cleaning", "r_scripts", "grouper-functions.R"))


## Load Mapping Data

In [8]:
proc <- fread(here(path_to_excel, "proc.csv"))
proc[, CODE := as.character(CODE)]
# head(proc)

rvs_icd9 <- fread(here(path_to_aux, "rvs_icd9cm.csv"),
  select = c("rvs", "icd9cm")
)
rvs_icd9[, rvs := as.character(rvs)]
rvs_icd9[, icd9cm := as.character(icd9cm * 100)]
rvs_icd9 <- merge(rvs_icd9, proc[, .(CODE, DRGUSE)],
  by.x = "icd9cm", by.y = "CODE", all.x = TRUE
)
rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE]
rvs_icd9 <- rvs_icd9[!is.na(rvs) & !is.na(icd9cm), -"DRGUSE"]
# head(rvs_icd9)

acr_rvs <- fread(here(path_to_aux, "acr_rvs.csv"))
# head(acr_rvs)

# Read in the data.table
tdrg_icd10 <- fread(here(path_to_aux, "i10.csv"))

# Set the key if not already set
setkey(tdrg_icd10, "CODE")
# head(tdrg_icd10)

# Subset and assign the result to acc_pdx
acc_pdx <- tdrg_icd10[ACCPDX == "Y", CODE]

# Optional: if CODEs are not unique in tdrg_icd10
acc_pdx <- unique(acc_pdx)


## Read Data

### Reading Data

In [9]:
options(verbose = FALSE)
options(warn = -1)

dt <- main_read_function()

if (file.exists(total_rows_file)) {
  total_rows <- readRDS(total_rows_file)
} else {
  total_rows <- fread(full_claims, select = 1L, header = TRUE)[, .N]
  saveRDS(total_rows, file = total_rows_file)
}


[1] "Sampled file exists. Reading the sampled file..."
[1] "Sampled file matches sample size."


In [10]:
options(warn = 1)


## Data Processing

### Data Cleaning

#### Chunking

In [11]:
if (to_chunk) {
  tic("Total execution time:")
  num_cores <- max(1, availableCores() - 1)

  result <- parallelize_and_summarize(
    dt, num_cores, to_view_checks, global_seed,
    rows_to_show, rvs_icd9, tdrg_icd10, acc_pdx
  )

  dt <- result$dt
  consolidated_summary <- result$consolidated_summary
  aggregate_statistics <- result$aggregate_statistics

  # Print the consolidated summary
  print(consolidated_summary)

  # Print the aggregated summary statistics
  cat(
    sprintf(
      "There are %d RVS codes without an ICD-9CM",
      aggregate_statistics$without_drg_count
    ), "equivalent recognized by the TDRG ICD9CM\n"
  )
  cat(
    sprintf(
      "There are %d unique RVS codes that appear in the claims.\n",
      aggregate_statistics$total_rvs_count
    )
  )
  cat(
    sprintf(
      "Of these, %d (%.2f%%) have a mapping to an ICD-9-CM code.\n",
      aggregate_statistics$mappable_rvs_count,
      aggregate_statistics$mappable_rvs_percentage
    )
  )
  cat(
    sprintf(
      "Of these, there are %d (%.2f%%)",
      aggregate_statistics$multi_mapped_rvs_count,
      aggregate_statistics$multi_mapped_rvs_percentage
    ), "with more than one ICD9 equivalent",
    "recognized by the Thai ICD9 library.\n"
  )
  cat(
    sprintf(
      "There are %d (%.2f%%) with no ICD-9-CM equivalents.\n\n\n",
      aggregate_statistics$unmappable_rvs_count,
      aggregate_statistics$unmappable_rvs_percentage
    )
  )
  cat(
    sprintf(
      "There are %d unique entries for ICD-10 codes, of which %d (%.2f%%)",
      aggregate_statistics$total_unique_icd_count,
      aggregate_statistics$direct_match_count,
      aggregate_statistics$direct_match_percentage
    ), "are directly in the Thai ICD-10 library\n"
  )
  cat(
    sprintf(
      "The modifications led to a total of %d codes being mapped to an equivalent in the Thai ICD10 library.\n",
      aggregate_statistics$total_mapped_count
    )
  )
  cat(
    sprintf(
      "Out of these, %d were modified to match.\n",
      aggregate_statistics$modified_count
    )
  )
  cat(
    sprintf(
      "There are %d codes that could not be mapped to the Thai ICD10 library:\n",
      aggregate_statistics$unmapped_icd_count
    )
  )
  print(head(aggregate_statistics$combined_unmapped_icds, rows_to_show))
}


[1] "Viewing checks"


There are 636 unique entries for ICD-10 codes, of which 531 (83.49%)  are directly in the Thai ICD-10 library
The modifications led to a total of 597 codes being mapped to an equivalent in the Thai ICD10 library.
Out of these, 66 were modified to match.
There are 39 codes that could not be mapped to the Thai ICD10 library:


Table: Unmapped ICD Codes

|code    |source   | count|
|:-------|:--------|-----:|
|NSD01   |clin_c1  |    73|
|MCP01   |clin_c1  |    46|
|A971    |clin_c1  |    27|
|A970    |clin_icd |    17|
|FP001   |clin_c1  |     9|
|Y95     |clin_icd |     7|
|0800    |clin_icd |     7|
|0809    |clin_icd |     5|
|Z49     |clin_icd |     3|
|W54     |clin_icd |     2|
|N08     |clin_icd |     2|
|V8999   |clin_icd |     2|
|A972    |clin_c1  |     1|
|ANC02   |clin_c1  |     1|
|ANC01   |clin_c1  |     1|
|Y841    |clin_icd |     1|
|W01     |clin_icd |     1|
|080     |clin_icd |     1|
|X5999   |clin_icd |     1|
|J17     |clin_icd |     1|
|Y459  

### Clean Data, Map codes, and Find PDx (if not chunking)

#### Clean Data

In [12]:
if (!to_chunk) {
  if (to_profvis) {
    tic("Total execution time:")
    p <- profvis({
      dt <- clean_data(dt)
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "clean_data.html"),
      selfcontained = TRUE
    )
  } else {
    tic("Total execution time:")
    dt <- clean_data(dt)
  }
}


#### Map Codes

In [13]:
# Map RVS codes
if (!to_chunk) {
  if (to_profvis) {
    p <- profvis({
      dt[, icd9_list := map_rvs_icd9(clin_rvs, rvs_icd9)]
      map_then_compare_icd_mappings(
        tdrg_icd10,
        rows_to_show,
        invalid_rows_to_show = rows_to_show
      )

      # Replace empty strings in character and factor columns with NA
      # dt <- replace_empty_with_na(dt, to_view_checks)
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "map_rvs.html"),
      selfcontained = TRUE
    )
  } else {
    dt[, icd9_list := map_rvs_icd9(clin_rvs, rvs_icd9)]
    map_then_compare_icd_mappings(
      tdrg_icd10,
      rows_to_show,
      invalid_rows_to_show = rows_to_show
    )

    # Replace empty strings in character and factor columns with NA
    # dt <- replace_empty_with_na(dt, to_view_checks)
  }
}


#### Find PDx

In [14]:
if (!to_chunk) {
  if (to_profvis) {
    p <- profvis({
      # dt <- apply_find_pdx(dt)
      pdx_result <- apply_find_pdx(dt$clin_c1, dt$clin_c2, dt$clin_icd, acc_pdx)
      dt$pdx <- pdx_result$pdx
      dt$pdx_code <- pdx_result$pdx_code
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "find_pdx.html"),
      selfcontained = TRUE
    )
  } else {
    # dt <- apply_find_pdx(dt)
    pdx_result <- apply_find_pdx(dt$clin_c1, dt$clin_c2, dt$clin_icd, acc_pdx)
    dt$pdx <- pdx_result$pdx
    dt$pdx_code <- pdx_result$pdx_code
  }
}


## Export

### Export Intermediate Output

In [15]:
if (to_write) {
  fwrite(dt, here(path_to_intermediate, paste0(
    "output_", year_to_load,
    suffix, ".csv"
  )))
}


### Export for Batch Grouper

In [16]:
if (to_group) {
  if (to_profvis) {
    p <- profvis({
      export_for_batch_grouper(dt, year_to_load, output_txt_file)
      for_batch_grouping <- fread(output_txt_file,
        sep = "|", na.strings = "--"
      )
      batch_grouping_result <- fread(grouper_result_file,
        sep = "|", na.strings = "--"
      )
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "export_for_grouper.html"),
      selfcontained = TRUE
    )
  } else {
    export_for_batch_grouper(dt, year_to_load, output_txt_file)
    for_batch_grouping <- fread(output_txt_file,
      sep = "|", na.strings = "--"
    )
    batch_grouping_result <- fread(grouper_result_file,
      sep = "|", na.strings = "--"
    )
  }
}


## Runtime Estimation

### Stop Timer

In [17]:
# Stop the timer and capture total time
toc_data <- toc(log = TRUE)
total_time <- toc_data$toc - toc_data$tic


Total execution time:: 10.41 sec elapsed


### Calculate Speed

In [18]:
# Calculate time spent per cell and per row
total_rows_dt <- nrow(dt)
total_cells <- nrow(dt) * ncol(dt)

time_per_cell <- total_time / total_cells
time_per_row <- total_time / total_rows_dt
time_estimate_total_rows <- time_per_row * total_rows

# Format the row numbers
formatted_total_rows_dt <- format_large_numbers(total_rows_dt)
formatted_total_rows <- format_large_numbers(total_rows)

# Print the results with aligned decimal points and formatted row numbers
cat(sprintf(
  "Time spent (total) for %2s rows:  %1.2f sec  (actual)\n",
  formatted_total_rows_dt, total_time
))
cat(sprintf(
  "Time spent (t/row) for %2s rows:  %1.2f msec (actual)\n",
  formatted_total_rows_dt, time_per_row * 1000
))
cat(sprintf(
  "Time spent (total) for  %2s rows: %2.2f min  (estimate)\n",
  formatted_total_rows, time_estimate_total_rows / 60
))


Time spent (total) for 25.0k rows:  10.41 sec  (actual)
Time spent (t/row) for 25.0k rows:  0.42 msec (actual)
Time spent (total) for  11.8m rows: 81.74 min  (estimate)
